# DataCo SMART Supply Chain Analytics
## Complete Exploratory Data Analysis & Business Insights

**Dataset:** DataCo SMART Supply Chain Dataset  
**Source:** DataCo Global  
**Rows:** 180,519 order-item records (2015–2018)  
**Columns:** 53 raw features

---
### Notebook Structure
1. Import Libraries
2. Load Dataset
3. Dataset Inspection
4. Data Quality Checks
5. Data Cleaning
6. Feature Engineering
7. Sales Analysis
8. Profitability Analysis
9. Customer Analysis
10. Product Analysis
11. Shipping & Delivery Analysis
12. Geographic Analysis
13. Business Questions Answered
14. Key Findings
15. Recommendations

## 1. Import Libraries

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Make sure analysis.py is importable when notebook is run from its own directory
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
import analysis as an

# Display settings
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid', palette='Set2')
%matplotlib inline

print('Libraries loaded successfully.')

## 2. Load Dataset

In [ ]:
# Load raw data using analysis.py
raw_df = an.load_data()
print(f'Raw dataset shape: {raw_df.shape}')
print(f'Rows: {raw_df.shape[0]:,}  |  Columns: {raw_df.shape[1]}')
raw_df.head()

## 3. Dataset Inspection

In [ ]:
print('=== Column Names & Data Types ===')
print(raw_df.dtypes.to_string())

In [ ]:
print('=== Descriptive Statistics — Numeric Columns ===')
raw_df.describe().round(2)

In [ ]:
print('=== Unique Value Counts for Key Categorical Columns ===')
cat_cols = ['Market','Order Region','Customer Segment','Shipping Mode',
            'Delivery Status','Order Status','Department Name','Category Name','Type']
for col in cat_cols:
    if col in raw_df.columns:
        vals = raw_df[col].dropna().unique()
        print(f'\n{col} ({len(vals)} unique):')
        print(' | '.join(sorted([str(v) for v in vals])))

## 4. Data Quality Checks

In [ ]:
# Missing values
missing = raw_df.isnull().sum()
missing_pct = (missing / len(raw_df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print('=== Missing Values ===')
print(missing_df.to_string())
print(f'\nTotal missing cells: {missing.sum():,}')

In [ ]:
# Duplicate rows
dupes = raw_df.duplicated().sum()
print(f'Duplicate rows: {dupes:,}')

# Negative sales
neg_sales = (raw_df['Sales'] < 0).sum()
print(f'Rows with negative Sales: {neg_sales:,}')

# Negative profit (expected — some orders are unprofitable)
neg_profit = (raw_df['Order Profit Per Order'] < 0).sum()
print(f'Rows with negative Profit: {neg_profit:,} ({neg_profit/len(raw_df)*100:.1f}%)')

# Date column check
raw_df['order date (DateOrders)'] = pd.to_datetime(
    raw_df['order date (DateOrders)'], format='%m/%d/%Y %H:%M', errors='coerce'
)
invalid_dates = raw_df['order date (DateOrders)'].isna().sum()
print(f'Invalid order dates: {invalid_dates:,}')
print(f'Date range: {raw_df["order date (DateOrders)"].min()} to {raw_df["order date (DateOrders)"].max()}')

## 5. Data Cleaning

In [ ]:
# Clean using analysis.py
raw_again = an.load_data()   # reload fresh
df_clean, clean_summary = an.clean_data(raw_again)

print('=== Cleaning Summary ===')
for k, v in clean_summary.items():
    print(f'  {k}: {v}')

print(f'\nShape after cleaning: {df_clean.shape}')
print(f'Remaining missing values: {df_clean.isnull().sum().sum():,}')

## 6. Feature Engineering

In [ ]:
# Apply feature engineering using analysis.py
df = an.engineer_features(df_clean)
print(f'Shape after feature engineering: {df.shape}')
print('\nNew columns added:')
new_cols = ['Order Year','Order Month','Order Month Name','Order Quarter',
            'Order YearMonth','Shipping Delay Days','Is Late','Profit Margin (%)','Revenue Band']
print(df[new_cols].head(5).to_string())

In [ ]:
# Orders per year
print('Orders by Year:')
print(df['Order Year'].value_counts().sort_index())

## 7. Sales Analysis

In [ ]:
# Top-level KPIs
kpis = an.get_kpis(df)
print('=== Key Performance Indicators ===')
for k, v in kpis.items():
    print(f'  {k}: {v}')

In [ ]:
sales = an.sales_analysis(df)

# Monthly trend
fig, ax = plt.subplots(figsize=(14, 4))
monthly = sales['monthly_trend']
ax.plot(monthly['Order YearMonth'], monthly['Total Sales'], marker='o', linewidth=1.5, color='steelblue')
ax.set_title('Monthly Revenue Trend', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Total Sales ($)')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

print('\nYearly Sales:')
print(sales['yearly_trend'].to_string(index=False))

In [ ]:
# Sales by market
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

mkt = sales['by_market'].sort_values('Total Sales', ascending=False)
axes[0].bar(mkt['Market'], mkt['Total Sales'] / 1e6, color=sns.color_palette('Set2', len(mkt)))
axes[0].set_title('Revenue by Market ($M)')
axes[0].set_ylabel('Sales ($M)')

seg = sales['by_segment'].sort_values('Total Sales', ascending=False)
axes[1].pie(seg['Total Sales'], labels=seg['Customer Segment'], autopct='%1.1f%%',
            colors=sns.color_palette('Pastel1', len(seg)))
axes[1].set_title('Revenue by Customer Segment')

plt.tight_layout()
plt.show()

## 8. Profitability Analysis

In [ ]:
profit = an.profit_analysis(df)

# Monthly profit trend
fig, ax = plt.subplots(figsize=(14, 4))
monthly_p = profit['monthly_trend']
colors_p = ['#d62728' if v < 0 else '#2ca02c' for v in monthly_p['Total Profit']]
ax.bar(monthly_p['Order YearMonth'], monthly_p['Total Profit'], color=colors_p)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Monthly Profit / Loss Trend')
ax.set_ylabel('Total Profit ($)')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# Profit margin by market
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

pm = profit['margin_by_market'].sort_values('Avg Profit Margin (%)', ascending=False)
axes[0].bar(pm['Market'], pm['Avg Profit Margin (%)'],
            color=sns.color_palette('RdYlGn', len(pm)))
axes[0].set_title('Avg Profit Margin (%) by Market')
axes[0].set_ylabel('Margin (%)')

dept_p = profit['by_department'].head(10)
axes[1].barh(dept_p['Department Name'], dept_p['Total Profit'] / 1e6,
             color=sns.color_palette('Blues_r', len(dept_p)))
axes[1].set_title('Top 10 Departments by Profit ($M)')
axes[1].set_xlabel('Total Profit ($M)')

plt.tight_layout()
plt.show()

In [ ]:
# Discount rate vs profit (scatter)
sample = df.sample(min(5000, len(df)), random_state=42)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(sample['Order Item Discount Rate'], sample['Order Profit Per Order'],
           alpha=0.15, s=10, color='steelblue')
ax.axhline(0, color='red', linewidth=1, linestyle='--')
ax.set_xlabel('Discount Rate')
ax.set_ylabel('Profit Per Order ($)')
ax.set_title('Discount Rate vs Profit Per Order')
plt.tight_layout()
plt.show()

## 9. Customer Analysis

In [ ]:
customers = an.customer_analysis(df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Segment order count
seg_c = customers['segment_counts']
axes[0].pie(seg_c['Order Items'], labels=seg_c['Customer Segment'], autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(seg_c)))
axes[0].set_title('Order Items by Segment')

# Sales by segment
seg_s = customers['sales_by_segment']
axes[1].bar(seg_s['Customer Segment'], seg_s['Total Sales'] / 1e6,
            color=sns.color_palette('Pastel1', len(seg_s)))
axes[1].set_title('Revenue by Segment ($M)')
axes[1].set_ylabel('Sales ($M)')

# Payment type
pay = customers['payment_type_counts']
axes[2].bar(pay['Type'], pay['Count'],
            color=sns.color_palette('Set3', len(pay)))
axes[2].set_title('Orders by Payment Type')

plt.tight_layout()
plt.show()

In [ ]:
# Top 20 customers
top_cust = customers['top_customers']
print(f'Total unique customers: {df["Customer Id"].nunique():,}')
print(f'Top customer by revenue: {top_cust.iloc[0]["Customer Name"]} — ${top_cust.iloc[0]["Total Sales"]:,.0f}')

## 10. Product Analysis

In [ ]:
products = an.product_analysis(df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 10 categories by sales
cat_s = products['sales_by_category'].head(10)
axes[0].barh(cat_s['Category Name'], cat_s['Total Sales'] / 1e6,
             color=sns.color_palette('Blues_r', 10))
axes[0].set_title('Top 10 Categories by Revenue ($M)')
axes[0].set_xlabel('Total Sales ($M)')

# Top 10 categories by profit
cat_p = products['profit_by_category'].head(10)
axes[1].barh(cat_p['Category Name'], cat_p['Total Profit'] / 1e6,
             color=sns.color_palette('Greens_r', 10))
axes[1].set_title('Top 10 Categories by Profit ($M)')
axes[1].set_xlabel('Total Profit ($M)')

plt.tight_layout()
plt.show()

In [ ]:
# Sales by department
dept_s = products['sales_by_department']
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(dept_s['Department Name'], dept_s['Total Sales'] / 1e6,
       color=sns.color_palette('Spectral', len(dept_s)))
ax.set_title('Revenue by Department ($M)')
ax.set_ylabel('Total Sales ($M)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 11. Shipping & Delivery Analysis

In [ ]:
shipping = an.shipping_analysis(df)
delivery = an.delivery_analysis(df)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Shipping mode distribution
mode_c = shipping['mode_counts']
axes[0, 0].pie(mode_c['Order Items'], labels=mode_c['Shipping Mode'], autopct='%1.1f%%',
               colors=sns.color_palette('Set2', len(mode_c)))
axes[0, 0].set_title('Order Items by Shipping Mode')

# Late rate by mode
late_mode = shipping['mode_late_rate'].sort_values('Late Rate (%)', ascending=False)
axes[0, 1].bar(late_mode['Shipping Mode'], late_mode['Late Rate (%)'],
               color=sns.color_palette('Reds_r', len(late_mode)))
axes[0, 1].set_title('Late Delivery Rate by Shipping Mode (%)')
axes[0, 1].set_ylabel('Late Rate (%)')

# Delivery status distribution
ds = delivery['status_counts']
axes[1, 0].bar(ds['Delivery Status'], ds['Count'],
               color=sns.color_palette('Pastel1', len(ds)))
axes[1, 0].set_title('Delivery Status Distribution')
plt.setp(axes[1, 0].get_xticklabels(), rotation=20, ha='right', fontsize=9)

# Order status distribution
os_c = delivery['order_status_counts'].head(8)
axes[1, 1].barh(os_c['Order Status'], os_c['Count'],
                color=sns.color_palette('Blues_r', len(os_c)))
axes[1, 1].set_title('Order Status Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Late delivery rate by region (top 15)
late_reg = delivery['late_by_region'].head(15)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(late_reg['Order Region'], late_reg['Late Rate (%)'],
        color=sns.color_palette('RdYlGn_r', len(late_reg)))
ax.set_title('Late Delivery Rate by Region (%) — Top 15')
ax.set_xlabel('Late Rate (%)')
plt.tight_layout()
plt.show()

## 12. Geographic Analysis

In [ ]:
geo = an.geo_analysis(df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Sales by market
sm = geo['sales_by_market'].sort_values('Total Sales', ascending=True)
axes[0].barh(sm['Market'], sm['Total Sales'] / 1e6,
             color=sns.color_palette('Blues_r', len(sm)))
axes[0].set_title('Revenue by Market ($M)')
axes[0].set_xlabel('Total Sales ($M)')

# Profit by market
pm = geo['profit_by_market'].sort_values('Total Profit', ascending=True)
colors_m = ['#d62728' if v < 0 else '#2ca02c' for v in pm['Total Profit']]
axes[1].barh(pm['Market'], pm['Total Profit'] / 1e6, color=colors_m)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Profit by Market ($M)')
axes[1].set_xlabel('Total Profit ($M)')

plt.tight_layout()
plt.show()

In [ ]:
# Top 20 countries by revenue
countries = geo['sales_by_country']
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(countries['Order Country'], countries['Total Sales'] / 1e6,
        color=sns.color_palette('Spectral', len(countries)))
ax.set_title('Top 20 Countries by Revenue ($M)')
ax.set_xlabel('Total Sales ($M)')
plt.tight_layout()
plt.show()

## 13. Business Questions Answered

In [ ]:
bq = an.answer_business_questions(df)

questions = {
    'Which market generates the most revenue?':            bq['top_market_by_revenue'],
    'Which customer segment is most profitable?':          bq['most_profitable_segment'],
    'What shipping mode is used the most?':                bq['most_used_shipping_mode'],
    'What is the on-time delivery rate?':                  f"{bq['on_time_delivery_rate_pct']:.1f}%",
    'Which product category has the highest margin?':      bq['highest_margin_category'],
    'Which region has the highest late delivery rate?':    bq['highest_late_rate_region'],
    'What is the average shipping delay (days)?':          bq['avg_shipping_delay_days'],
    'What is the most common order status?':               bq['most_common_order_status'],
    'Which department drives the most sales?':             bq['top_department_by_sales'],
    'What is the average discount rate?':                  f"{bq['avg_discount_rate_pct']:.1f}%",
}

print('=== Business Questions & Answers ===')
for q, a in questions.items():
    print(f'  Q: {q}')
    print(f'  A: {a}\n')

## 14. Key Findings

In [ ]:
insights = an.generate_insights(df)
print('=== Key Findings ===')
for i, ins in enumerate(insights, 1):
    print(f'{i}. {ins}')

## 15. Recommendations

In [ ]:
recs = an.generate_recommendations(df)
print('=== Recommendations ===')
for r in recs:
    print(f"\n[{r['priority']} Priority]")
    print(f"  Finding:        {r['finding']}")
    print(f"  Recommendation: {r['recommendation']}")

---
### Notebook Summary
This notebook provides a complete EDA of the DataCo SMART Supply Chain Dataset covering:
- **180,519** order-item records across 2015–2018
- **$36.8M** total revenue across 5 global markets
- Sales, profitability, customer, product, shipping, and geographic analyses
- 10 business questions answered with computed values
- Actionable recommendations to improve delivery performance and profitability

All KPIs are computed dynamically via `analysis.py` — no hard-coded values.